#### Loading CSV

In [2]:
import pandas as pd

In [3]:
csv_path = "CorpFamily.csv"

In [4]:
df = pd.read_csv(csv_path)

In [5]:
df.head()

,Order,Company Name,Is Headquarters,Ownership Type,Entity Type,City,State Or Province,Country/Region,Employees (Single Site),Employees (Total),Sales (USD),D&B Hoovers Industry,Key ID,D-U-N-S® Number,D&B Hoovers Contacts,Direct Marketing Status
0,1,"Palo Alto Networks, Inc.",True,Public,Parent,Santa Clara,California,United States,500.0,13948.0,6.892700e+09,Software,111346086,198144771,8203,Has Not Opted Out of Direct Marketing
1,2,PALOALTO NETWORKS K.K.,True,Private,Subsidiary,Chiyoda-Ku,Tokyo,Japan,NaN,300.0,7.373802e+06,Data Processing,202620726,690720130,1,Has Not Opted Out of Direct Marketing
2,3,PALOALTO NETWORKS,False,Private,Branch,Osaka,Osaka,Japan,NaN,NaN,NaN,Data Processing,572435340,680080907,0,Has Not Opted Out of Direct Marketing
3,4,PALOALTO NETWORKS K.K.,False,Private,Branch,Nagoya,Aichi,Japan,NaN,NaN,NaN,Data Processing,252892609,693120357,0,Has Not Opted Out of Direct Marketing
4,5,"Palo Alto Networks International, Inc.",True,Private,Subsidiary,Santa Clara,California,United States,16.0,226.0,6.423768e+06,Computer and Peripheral Equipment Manufacturing,471626962,117297477,0,Has Not Opted Out of Direct Marketing


#### Loading JSON

In [11]:
import json

In [8]:
json_path = "familyTree.json"

In [9]:
with open(json_path,"r") as f:
    data = f.read()

In [12]:
data = json.loads(data)

In [13]:
data.keys()

dict_keys(['company'])

In [14]:
data["company"].keys()

dict_keys(['name', 'duns', 'city', 'state', 'country', 'phone', 'locationStatus', 'childNodeCount', 'locationStatusLabel', 'family', 'companies'])

In [21]:
_flatten = JsonFlatten()

In [22]:
data_flattened = _flatten.flatten_data(data["company"])

In [23]:
len(data_flattened)

78

In [24]:
pd.DataFrame(data_flattened)

,family_global_duns,family_parent_duns,family_domestic_duns,companies_name,companies_duns,companies_city,companies_state,companies_country,companies_phone,companies_location_status,...,companies_companies_companies_companies_companies_companies_companies_companies_country,companies_companies_companies_companies_companies_companies_companies_companies_phone,companies_companies_companies_companies_companies_companies_companies_companies_location_status,companies_companies_companies_companies_companies_companies_companies_companies_child_node_count,companies_companies_companies_companies_companies_companies_companies_companies_location_status_label,companies_companies_companies_companies_companies_companies_companies_companies_family_global_duns,companies_companies_companies_companies_companies_companies_companies_companies_family_parent_duns,companies_companies_companies_companies_companies_companies_companies_companies_family_domestic_duns,companies_companies_companies_companies_companies_companies_companies_companies_companies,companies_companies_companies_companies_companies_companies
0,198144771,198144771,198144771,Alto Palo Networks Inc,117699242,Milpitas,California,US,4085761985,BR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,198144771,198144771,198144771,Alto Palo Networks Inc,119030363,Sunnyvale,California,US,None,BR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,198144771,198144771,198144771,Palo Alto Networks Inc.,079389067,Santa Clara,California,US,None,BR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,198144771,198144771,198144771,"Cyvera, Inc",079096086,San Francisco,California,US,4155130201,SL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,198144771,198144771,198144771,"Demisto, Inc.",080258888,Cupertino,California,US,4089058344,SL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,198144771,198144771,198144771,"Palo Alto Networks (México), S. de R.L. de C.V.",812299351,México,CIUDAD DE MEXICO,MX,5539012236,SL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,198144771,198144771,198144771,"Palo Alto Networks Financial Services, LLC",118196090,Santa Clara,California,US,8668989087,SL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,198144771,198144771,198144771,Pan LLC,042961421,Pembroke Pines,Florida,US,9543095956,SL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,198144771,198144771,198144771,Alto Palo Networks Inc,068376681,Alviso,California,US,4087860001,BR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
list_of_companies = list()

In [50]:
def recursively_check_companies(record):
    if record.get("companies") is not None:
        list_of_companies.append(record)
        for t in record["companies"]:
            recursively_check_companies(t)
    else:
        list_of_companies.append(record)

In [51]:
recursively_check_companies(data["company"])

In [54]:
pd.DataFrame(list_of_companies).to_csv("./processed_json.csv")

#### Pydantic

In [27]:
from pydantic import BaseModel,Field
from typing import List

In [29]:
class CompanyBase(BaseModel):
    name: str
    duns: str
    city: str
    state: str
    country: str
    phone: str = Field(default=None) 
    locationStatus: str
    childNodeCount: str
    locationStatusLabel: str
    family: Family
        

In [30]:
class Family(BaseModel):
    globalDuns: str
    parentDuns: str
    domesticDumns: str
class Company(BaseModel):
    name: str
    duns: str
    city: str
    state: str
    country: str
    phone: str = Field(default=None) 
    locationStatus: str
    childNodeCount: str
    locationStatusLabel: str
    family: Family
    companies: List[CompanyBase]

In [33]:
Company.parse_obj(data["company"])

ValidationError: 44 validation errors for Company
family -> domesticDumns
  field required (type=value_error.missing)
companies -> 0 -> family -> domesticDumns
  field required (type=value_error.missing)
companies -> 1 -> family -> domesticDumns
  field required (type=value_error.missing)
companies -> 2 -> family -> domesticDumns
  field required (type=value_error.missing)
companies -> 3 -> family -> domesticDumns
  field required (type=value_error.missing)
companies -> 4 -> family -> domesticDumns
  field required (type=value_error.missing)
companies -> 5 -> family -> domesticDumns
  field required (type=value_error.missing)
companies -> 6 -> family -> domesticDumns
  field required (type=value_error.missing)
companies -> 7 -> family -> domesticDumns
  field required (type=value_error.missing)
companies -> 8 -> family -> domesticDumns
  field required (type=value_error.missing)
companies -> 9 -> family -> domesticDumns
  field required (type=value_error.missing)
companies -> 10 -> fam

### Importing and Initialising import functions

In [1]:
#https://stackoverflow.com/questions/51359783/how-to-flatten-multilevel-nested-json
#TODO Validate this code perfoermance and better it if needed

import inflection
from functools import (partial,
                       singledispatch)
from itertools import chain
from typing import (Dict,
                    List,
                    TypeVar)



def dict_snake_case(data: dict) -> dict:
    """
    This method converts keys of a dictionary to snake case
    example: {"AbCd":1,"A__C":2,"Ac_Del":3} output: {'ab_cd': 1, 'a__c': 2, 'ac_del': 3}
    """
    new_keys = [inflection.underscore(key) for key in data.keys()]
    return dict(zip(new_keys, data.values()))

Serializable = TypeVar('Serializable', None, int, bool, float, str,
                       dict, list, tuple)
Array = List[Serializable]
Object = Dict[str, Serializable]


def flatten(object_: Object,
            *,
            path_separator: str = '_') -> Array[Object]:
    """
    Flattens given JSON object into list of objects with non-nested values.

    """
    keys = set(object_)
    result = [dict(object_)]
    while keys:
        key = keys.pop()
        new_result = []
        for index, record in enumerate(result):
            try:
                value = record[key]
            except KeyError:
                new_result.append(record)
            else:
                if isinstance(value, dict):
                    del record[key]
                    new_value = flatten_nested_objects(
                            value,
                            prefix=key + path_separator,
                            path_separator=path_separator
                    )
                    keys.update(new_value.keys())
                    new_result.append({**new_value, **record})
                elif isinstance(value, list):
                    del record[key]
                    new_records = [
                        flatten_nested_objects(sub_value,
                                               prefix=key + path_separator,
                                               path_separator=path_separator)
                        for sub_value in value
                    ]
                    keys.update(chain.from_iterable(map(dict.keys,
                                                        new_records)))
                    if new_records:
                        new_result.extend({**new_record, **record}
                                          for new_record in new_records)
                    else:
                        new_result.append(record)
                else:
                    new_result.append(record)
        result = new_result
    return result


@singledispatch
def flatten_nested_objects(object_: Serializable,
                           *,
                           prefix: str = '',
                           path_separator: str) -> Object:
    return {prefix[:-len(path_separator)]: object_}


@flatten_nested_objects.register(dict)
def _(object_: Object,
      *,
      prefix: str = '',
      path_separator: str) -> Object:
    result = dict(object_)
    for key in list(result):
        result.update(flatten_nested_objects(result.pop(key),
                                             prefix=(prefix + key
                                                     + path_separator),
                                             path_separator=path_separator))
    return result


@flatten_nested_objects.register(list)
def _(object_: Array,
      *,
      prefix: str = '',
      path_separator: str) -> Object:
    return {prefix[:-len(path_separator)]: list(map(partial(
            flatten_nested_objects,
            path_separator=path_separator),
            object_))}
class JsonFlatten():
    def __check_dtypes(self, cell):
        if isinstance(cell, list):
            print("Needs more flattening, {} {}".format(len(cell),"-".join(cell[:2])))
            raise ValueError("LIST - Unflattened")

        elif isinstance(self, dict):
            print("This is a dictionary. Not completely flattened yet. {}".format("-".join(list(cell.keys()))))
            raise ValueError("dict - Unflattened")
        else:
            pass

    def __dict_snake_case(self,data: dict) -> dict:
        """
        This method converts keys of a dictionary to snake case
        example: {"AbCd":1,"A__C":2,"Ac_Del":3}
        """
        new_keys = [inflection.underscore(key) for key in data.keys()]
        return dict(zip(new_keys, data.values()))

    def __adding_extra_columns(self, flattened_row, **extra_columns):
        processed_row = []
        for row in flattened_row:
            for cell in row:
                _ = self.__check_dtypes(row[cell])
            row = self.__dict_snake_case(row)
            if len(extra_columns) > 0:
                for extra_column in extra_columns:
                    row[extra_column] = extra_columns[extra_column]
            processed_row.append(row)
        return processed_row
    def convert_fields_to_json(self,data,cols_to_convert: list):
        if len(cols_to_convert) == 0:
            pass
        else:
            for col in cols_to_convert:
                try:
                    data[col] = json.dumps(data[col])
                except Exception as e:
                    logger.warning(f"Error converting the col to json: {str(e)}")
                    logger.debug(f"Debug of error: {data}",exc_info=True)

        return data
    def flatten_data(self, data, cols_to_convert = [],**extra_columns):
        msg = []
        if isinstance(data,dict):
            data = self.convert_fields_to_json(data=data,cols_to_convert=cols_to_convert)
            flattened_row = flatten(data)
            processed_row = self.__adding_extra_columns(flattened_row, **extra_columns)
            msg.extend(processed_row)
        elif isinstance(data,list) or isinstance(data,types.GeneratorType):
            for e in data:
                e = self.convert_fields_to_json(data=e, cols_to_convert=cols_to_convert)
                flattened_row = flatten(e)
                processed_row = self.__adding_extra_columns(flattened_row, **extra_columns)
                msg.extend(processed_row)
        else:
            raise ValueError("Data is of dtype {}".format(type(data)))
        return msg